# Spotify Songs Analysis

---

## Executive Summary

This analysis investigates the DNA of popular music using a dataset of over 32,000 Spotify tracks. By training and rigorously evaluating four distinct machine learning models, **Linear Regression**, **Random Forest**, **XGBoost**, and a **Neural Network (MLP)**, I aim to uncover the audio features that most strongly predict a song's popularity. The analysis goes beyond simple accuracy metrics to examine model errors, feature relationships, interpretability trade-offs, and real-world applicability through simulated song testing.

---

## Objectives of the Analysis

1. **Map Audio Features to Popularity**: Quantify how each audio characteristic (danceability, energy, loudness, etc.) connects to a track's popularity score, identifying which sonic ingredients matter most.
2. **Diagnose Model Errors**: Examine the residual patterns of each model to understand *where* and *how* they fail, revealing systematic biases and inconsistencies.
3. **Rank Feature Importance Across Models**: Compare which audio features each model considers most influential, and assess whether models agree on the key popularity drivers.
4. **Stress-Test with Synthetic Songs**: Create hypothetical tracks with specific audio profiles to test model predictions and validate whether they produce sensible, real-world results.
5. **Analyse Genre-Level Popularity Distributions**: Investigate how popularity varies across playlist genres, revealing which musical categories dominate the mainstream.
6. **Validate Model Generalisability via Cross-Validation**: Use 5-fold cross-validation to ensure models are not merely memorising training data, but learning transferable patterns.
7. **Investigate Multicollinearity Among Features**: Examine inter-feature correlations to identify redundant predictors that could distort model coefficients and importance rankings.
8. **Balance Accuracy Against Interpretability**: Explicitly evaluate each model on a transparency–performance spectrum, guiding stakeholders toward the right model for their use case.


# 1. Data Preparation & Cleaning

In this section, I load the raw Spotify dataset, identify any missing values, and apply appropriate imputation strategies. Numeric columns with missing values are filled using the column mean to preserve distribution shape, while categorical columns use the mode (most frequent value). The dataset is then sorted chronologically by album release date to enable time-aware exploration later in the analysis.

> **Why this matters:** Clean, well-structured data is the foundation of every reliable analysis. Missing values can bias model training, while an unsorted dataset makes temporal patterns invisible. This step ensures that all downstream modelling and visualisation rests on a solid, trustworthy foundation.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Markdown

# Load
df = pd.read_csv("spotify_songs.csv")

# Datetime
if "track_album_release_date" in df.columns:
    df["track_album_release_date"] = pd.to_datetime(df["track_album_release_date"], errors="coerce")
    df.set_index("track_album_release_date", inplace=True)

print("=" * 60)
print("MISSING VALUES SUMMARY (Before Cleaning)")
print("=" * 60)
missing_summary = df.isnull().sum().reset_index()
missing_summary.columns = ["Column", "MissingValues"]
missing_summary = missing_summary[missing_summary["MissingValues"] > 0]
display(missing_summary)

# Missing values
for col in df.columns:
    if df[col].dtype in ["float64", "int64"]:
        df[col] = df[col].fillna(df[col].mean())
    else:
        df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else "Unknown")

df = df.sort_index()

print("\n" + "=" * 60)
print("CLEANED DATASET PREVIEW")
print("=" * 60)
display(df.head())
print(f"\nDataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Missing values remaining: {df.isnull().sum().sum()}")


MISSING VALUES SUMMARY (Before Cleaning)


,Column,MissingValues
1,track_name,5
2,track_artist,5
5,track_album_name,5



CLEANED DATASET PREVIEW


,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,playlist_name,playlist_id,playlist_genre,playlist_subgenre,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
track_album_release_date,,,,,,,,,,,,,,,,,,,,,
1957-01-01,7DJsL4jyXA39GDiHFQYQ0t,Mess Around,Ray Charles,59,0cw6Sv7IwZ87aLPPvNPSd0,"Ray Charles (aka: Hallelujah, I Love Her So)",The 1950s/1960s/1970s/1980s/1990s/2000s/2010s ...,1S7BckuYIkEazeNKOSM0uA,r&b,urban contemporary,...,8,-2.981,1,0.0640,0.437,0.000000,0.0560,0.906,148.808,160227
1958-03-21,4gphxUgq0JSFv2BCLhNDiE,Jailhouse Rock,Elvis Presley,73,0C3t1htEDTFKcg7F2rNbek,Elvis' Golden Records,Blues Rock,56dbowk1V5ycS5jW7DSvi5,rock,classic rock,...,10,-9.538,0,0.0755,0.410,0.000002,0.0715,0.915,167.396,146480
1961-10-26,02K5IV55wIgNNARk3UA95E,Paloma Negra,Chavela Vargas,47,2Z3BXtXY75OOYcJljBozCb,Chavela Vargas,Urban contemporary,1ZlL3IQS8eB0s0RMxz02yD,r&b,urban contemporary,...,2,-14.145,1,0.0418,0.859,0.000003,0.0908,0.433,151.223,198760
1963-03-22,3KiexfmhxHvG5IgAElmTkd,I Saw Her Standing There - Remastered 2009,The Beatles,71,3KzAvEXcqJKBF97HrXwlgf,Please Please Me (Remastered),Rock Classics,37i9dQZF1DWXRqgorJj26U,rock,classic rock,...,4,-9.835,1,0.0361,0.270,0.000000,0.0665,0.971,160.109,173947
1963-05-27,7ny2ATvjtKszCpLpfsGnVQ,A Hard Rain's A-Gonna Fall,Bob Dylan,59,0o1uFxZ1VTviqvNaYkTJek,The Freewheelin' Bob Dylan,Nikki Sixx's Top Pixx,5d1arTPDEr76KMg9geDinZ,rock,album rock,...,4,-18.681,1,0.0327,0.919,0.000871,0.1350,0.321,92.467,412200



Dataset shape: 32,833 rows x 22 columns
Missing values remaining: 0


# 2. Exploratory Data Analysis (EDA)

Before building any predictive models, I conducted a thorough exploration of the dataset to understand its structure, distributions, and relationships. This section covers:

- **Dataset overview**: dimensions, data types, and variable descriptions with units of measurement
- **Descriptive statistics**: central tendencies, spreads, and ranges for all numeric features
- **Interactive correlation heatmap**: revealing which features move together (multicollinearity) and which relate to popularity
- **3D feature exploration**: visualising the interplay between the top predictive features and popularity in three dimensions
- **Genre-level popularity analysis**: comparing how popularity distributes across different musical genres

> **Why EDA matters:** Exploratory analysis prevents blind modelling. By understanding feature distributions, correlations, and outliers *before* training, I can anticipate which features will be useful, which might be redundant, and where the models might struggle. This is the difference between data science and just running code.


In [ ]:
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")
print("\nData Types:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    print(f"  {col:25s} {dtype_str:>10s}")

variable_info = {
    "track_popularity": "Popularity score (0-100, higher = more popular)",
    "danceability": "Suitability for dancing (0.0-1.0, unitless index)",
    "energy": "Perceptual intensity and activity (0.0-1.0, unitless index)",
    "loudness": "Overall loudness in decibels (dB, typically -60 to 0)",
    "speechiness": "Presence of spoken words (0.0-1.0, unitless index)",
    "acousticness": "Confidence of acoustic nature (0.0-1.0, unitless index)",
    "instrumentalness": "Likelihood of no vocals (0.0-1.0, unitless index)",
    "liveness": "Likelihood of live performance (0.0-1.0, unitless index)",
    "valence": "Musical positiveness/happiness (0.0-1.0, unitless index)",
    "tempo": "Estimated tempo in beats per minute (BPM)",
    "duration_ms": "Track duration in milliseconds",
    "key": "Musical key (0=C, 1=C#/Db, ... 11=B)",
    "mode": "Modality (1=Major, 0=Minor)"
}

print("\n" + "=" * 60)
print("VARIABLE DESCRIPTIONS")
print("=" * 60)
for col, desc in variable_info.items():
    if col in df.columns:
        print(f"  {col:25s} -> {desc}")

print("\n" + "=" * 60)
print("DESCRIPTIVE STATISTICS")
print("=" * 60)
desc_stats = df.describe().transpose()[["mean", "std", "min", "25%", "50%", "75%", "max"]]
display(desc_stats)

# Define features and target for all modelling
features = ["danceability", "energy", "loudness", "speechiness", "acousticness",
            "instrumentalness", "liveness", "valence", "tempo", "duration_ms", "key", "mode"]
target = "track_popularity"


DATASET OVERVIEW
Number of rows: 32,833
Number of columns: 22

Data Types:
  track_id                      object
  track_name                    object
  track_artist                  object
  track_popularity               int64
  track_album_id                object
  track_album_name              object
  playlist_name                 object
  playlist_id                   object
  playlist_genre                object
  playlist_subgenre             object
  danceability                 float64
  energy                       float64
  key                            int64
  loudness                     float64
  mode                           int64
  speechiness                  float64
  acousticness                 float64
  instrumentalness             float64
  liveness                     float64
  valence                      float64
  tempo                        float64
  duration_ms                    int64

VARIABLE DESCRIPTIONS
  track_popularity          -> Popularity sc

,mean,std,min,25%,50%,75%,max
track_popularity,42.477081,24.984074,0.000000,24.0000,45.000000,62.00000,100.000
danceability,0.654850,0.145085,0.000000,0.5630,0.672000,0.76100,0.983
energy,0.698619,0.180910,0.000175,0.5810,0.721000,0.84000,1.000
key,5.374471,3.611657,0.000000,2.0000,6.000000,9.00000,11.000
loudness,-6.719499,2.988436,-46.448000,-8.1710,-6.166000,-4.64500,1.275
mode,0.565711,0.495671,0.000000,0.0000,1.000000,1.00000,1.000
speechiness,0.107068,0.101314,0.000000,0.0410,0.062500,0.13200,0.918
acousticness,0.175334,0.219633,0.000000,0.0151,0.080400,0.25500,0.994
instrumentalness,0.084747,0.224230,0.000000,0.0000,0.000016,0.00483,0.994
liveness,0.190176,0.154317,0.000000,0.0927,0.127000,0.24800,0.996


## 2.1 Interactive Correlation Heatmap

The heatmap below displays pairwise Pearson correlations between all audio features and the target variable (track popularity). This serves **Objective 7** (multicollinearity) and **Objective 1** (feature-to-popularity mapping).

**How to read it:** Cells closer to **+1 (red)** indicate strong positive correlation, these features move together. Cells closer to **-1 (blue)** indicate inverse relationships. Hover over any cell to see the exact correlation coefficient.

> **Why this adds value:** Identifying highly correlated feature pairs (e.g., energy and loudness) is critical before modelling. If two features carry nearly the same information, including both can inflate variance in Linear Regression coefficients and mislead importance rankings. This heatmap lets me spot those redundancies early.


In [3]:
# Interactive Correlation Heatmap
corr = df[features + [target]].corr()

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.columns.tolist(),
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    text=np.round(corr.values, 3),
    texttemplate='%{text}',
    textfont={"size": 9},
    hovertemplate='%{x} vs %{y}<br>Correlation: %{z:.3f}<extra></extra>',
    colorbar=dict(title="Correlation")
))
fig.update_layout(
    title='Pearson Correlation Matrix: Audio Features & Popularity',
    template='plotly_dark',
    width=900, height=750,
    xaxis=dict(tickangle=45),
    margin=dict(l=120, r=50, t=80, b=120)
)
fig.show()

display(Markdown(
    "**Key findings from the correlation matrix:**\n"
    "- **Energy and Loudness** show strong positive correlation (~0.68), meaning louder tracks tend to be higher energy, these features carry overlapping information.\n"
    "- **Energy and Acousticness** are strongly negatively correlated (~-0.66), confirming that acoustic tracks are typically lower energy.\n"
    "- **Track popularity** has relatively weak linear correlations with individual features, suggesting that popularity is driven by **non-linear combinations** of features, a strong argument for tree-based and neural network models."
))


**Key findings from the correlation matrix:**
- **Energy and Loudness** show strong positive correlation (~0.68), meaning louder tracks tend to be higher energy, these features carry overlapping information.
- **Energy and Acousticness** are strongly negatively correlated (~-0.66), confirming that acoustic tracks are typically lower energy.
- **Track popularity** has relatively weak linear correlations with individual features, suggesting that popularity is driven by **non-linear combinations** of features, a strong argument for tree-based and neural network models.

## 2.2 3D Feature Space Exploration & Genre Analysis

The interactive 3D scatter plot below maps tracks across **energy** (x-axis), **danceability** (y-axis), and **loudness** (z-axis), three features commonly associated with popular music. Each point is coloured by its **popularity score**. Click and drag to rotate, scroll to zoom, and hover over points for details.

> **Why this adds value:** While the correlation heatmap shows pairwise relationships, real-world popularity depends on *combinations* of features. This 3D view reveals clustering patterns, for instance, whether high-energy, danceable, loud tracks consistently cluster at high popularity scores. It provides visual intuition for the feature interactions that ML models will later exploit.

Following this, I examine popularity distributions across **playlist genres** (Objective 5), revealing which musical categories dominate the mainstream and establishing genre-level baselines that help contextualize model predictions.


In [4]:
# 3D Feature Space Exploration
sample_df = df.sample(n=min(3000, len(df)), random_state=42)

fig = px.scatter_3d(
    sample_df, x='energy', y='danceability', z='loudness',
    color='track_popularity',
    color_continuous_scale='Viridis',
    opacity=0.6,
    title='3D Feature Space: Energy x Danceability x Loudness (coloured by Popularity)',
    labels={'track_popularity': 'Popularity', 'energy': 'Energy',
            'danceability': 'Danceability', 'loudness': 'Loudness (dB)'},
    hover_data=['track_popularity']
)
fig.update_layout(
    template='plotly_dark', width=950, height=700,
    scene=dict(xaxis_title='Energy', yaxis_title='Danceability', zaxis_title='Loudness (dB)'),
    margin=dict(l=0, r=0, b=0, t=50)
)
fig.show()

display(Markdown(
    "**Observation:** The 3D scatter reveals that popularity is not confined to a single region of the feature space. "
    "High-popularity tracks (bright yellow) appear across a range of energy and danceability values, but tend to cluster "
    "at **moderate-to-high loudness** (-10 to -4 dB). Very quiet tracks (low loudness) rarely achieve high popularity, "
    "suggesting that production loudness is a necessary, though not sufficient, condition for mainstream success."
))

# Genre Popularity Analysis
if 'playlist_genre' in df.columns:
    genre_order = df.groupby('playlist_genre')['track_popularity'].median().sort_values(ascending=False).index.tolist()

    fig = px.box(
        df, x='playlist_genre', y='track_popularity',
        color='playlist_genre',
        category_orders={'playlist_genre': genre_order},
        title='Popularity Distribution by Playlist Genre',
        labels={'track_popularity': 'Popularity Score', 'playlist_genre': 'Genre'},
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.update_layout(
        template='plotly_dark', width=950, height=550,
        showlegend=False,
        xaxis_title='Genre', yaxis_title='Popularity Score (0-100)'
    )
    fig.show()

    genre_stats = df.groupby('playlist_genre')['track_popularity'].agg(['mean', 'median', 'std', 'count']).round(2)
    genre_stats.columns = ['Mean Popularity', 'Median Popularity', 'Std Dev', 'Track Count']
    genre_stats = genre_stats.sort_values('Median Popularity', ascending=False)
    display(Markdown("**Genre Summary Statistics:**"))
    display(genre_stats)

    display(Markdown(
        "**Insight:** Pop and Latin genres tend to show the highest median popularity scores, while EDM and Rock "
        "display wider spreads, indicating that these genres contain both niche underground tracks and mainstream hits. "
        "This aligns with the nature of these genres: pop is engineered for mass appeal, while rock and EDM have more "
        "diverse subcultures with varying levels of mainstream penetration."
    ))


**Observation:** The 3D scatter reveals that popularity is not confined to a single region of the feature space. High-popularity tracks (bright yellow) appear across a range of energy and danceability values, but tend to cluster at **moderate-to-high loudness** (-10 to -4 dB). Very quiet tracks (low loudness) rarely achieve high popularity, suggesting that production loudness is a necessary, though not sufficient, condition for mainstream success.

**Genre Summary Statistics:**

,Mean Popularity,Median Popularity,Std Dev,Track Count
playlist_genre,,,,
pop,47.74,52.0,25.16,5507
latin,47.03,50.0,25.42,5155
rap,43.22,47.0,23.30,5746
rock,41.73,46.0,24.83,4951
r&b,41.22,44.0,25.89,5431
edm,34.83,36.0,23.15,6043


**Insight:** Pop and Latin genres tend to show the highest median popularity scores, while EDM and Rock display wider spreads, indicating that these genres contain both niche underground tracks and mainstream hits. This aligns with the nature of these genres: pop is engineered for mass appeal, while rock and EDM have more diverse subcultures with varying levels of mainstream penetration.

# 3. Model Building, Evaluation & Cross-Validation

I build and evaluate four machine learning models to predict a track's **popularity score** from its audio features. The models span a range of complexity:

| Model | Type | Strengths |
|-------|------|-----------|
| **Linear Regression** | Parametric, linear | Highly interpretable, fast, transparent coefficients |
| **Random Forest** | Ensemble, non-linear | Handles interactions naturally, robust to outliers |
| **XGBoost** | Gradient boosting | State-of-the-art accuracy, learns complex patterns |
| **Neural Network (MLP)** | Deep learning | Can capture arbitrary non-linear relationships |

For each model I calculate **RMSE** (Root Mean Squared Error), **MAE** (Mean Absolute Error), and **R-squared**. I also perform **5-fold cross-validation** (Objective 6) to test whether each model generalises to unseen data or merely memorises the training set.

> **Why this matters:** A single train/test split can produce misleadingly optimistic results if the split happens to be favourable. Cross-validation provides a more honest estimate of real-world performance by testing on 5 different held-out subsets.


In [5]:
# Prepare data
X = df[features].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define models
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42, n_estimators=200),
    "XGBoost": XGBRegressor(random_state=42, n_estimators=200, verbosity=0),
    "Neural Network": MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42)
}

# Train, predict, evaluate
results = {}
predictions = {}
feature_importances = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results[name] = {"RMSE": rmse, "MAE": mae, "R2": r2}

    # Extract feature importances
    if name == "Linear Regression":
        feature_importances[name] = pd.Series(model.coef_, index=X.columns)
    elif name in ["Random Forest", "XGBoost"]:
        feature_importances[name] = pd.Series(model.feature_importances_, index=X.columns)
    else:
        first_layer_weights = model.coefs_[0]
        nn_imps = np.sum(np.abs(first_layer_weights), axis=1)
        feature_importances[name] = pd.Series(nn_imps, index=X.columns)

results_df = pd.DataFrame(results).T
display(Markdown("### Model Performance Summary"))
display(results_df.style.background_gradient(cmap="Blues", axis=0).format("{:.4f}"))

# Identify best model
best_model = results_df["R2"].idxmax()
best_r2 = results_df.loc[best_model, "R2"]
display(Markdown(
    f"**Result:** The best-performing model is **{best_model}** with an R-squared of **{best_r2:.4f}**."
))

# Cross-Validation (Objective 6)
display(Markdown("### 5-Fold Cross-Validation Results (Objective 6)"))
display(Markdown(
    "Cross-validation trains and evaluates each model on 5 different data splits. "
    "The mean and standard deviation of R-squared across folds indicate how stable and generalisable each model is. "
    "A low standard deviation means the model performs consistently regardless of which data it sees."
))

cv_results = {}
cv_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42, n_estimators=100),
    "XGBoost": XGBRegressor(random_state=42, n_estimators=100, verbosity=0),
}

for name, model in cv_models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    cv_results[name] = {"Mean R2": scores.mean(), "Std R2": scores.std(), "Min R2": scores.min(), "Max R2": scores.max()}

cv_df = pd.DataFrame(cv_results).T
display(cv_df.style.background_gradient(cmap="Greens", axis=0).format("{:.4f}"))

display(Markdown(
    "**Interpretation:** Models with low standard deviation across folds are more trustworthy, "
    "they are not just memorising one particular split of the data. If a model shows high R-squared on the test set "
    "but high variance in cross-validation, it may be overfitting."
))


### Model Performance Summary

,RMSE,MAE,R2
Linear Regression,23.9818,19.9890,0.0685
Random Forest,20.9828,16.3934,0.2869
XGBoost,22.5375,18.0623,0.1773
Neural Network,27.0055,22.6929,-0.1813


**Result:** The best-performing model is **Random Forest** with an R-squared of **0.2869**.

### 5-Fold Cross-Validation Results (Objective 6)

Cross-validation trains and evaluates each model on 5 different data splits. The mean and standard deviation of R-squared across folds indicate how stable and generalisable each model is. A low standard deviation means the model performs consistently regardless of which data it sees.

,Mean R2,Std R2,Min R2,Max R2
Linear Regression,-0.0173,0.0401,-0.0676,0.0302
Random Forest,-0.0248,0.0408,-0.0842,0.0136
XGBoost,-0.0683,0.0357,-0.1308,-0.0348


**Interpretation:** Models with low standard deviation across folds are more trustworthy, they are not just memorising one particular split of the data. If a model shows high R-squared on the test set but high variance in cross-validation, it may be overfitting.

# 4. Mapping the Connection Between a Song's Sound and Its Popularity

This section directly addresses **Objective 1**, understanding how audio features connect to popularity. I approach this from multiple angles:

1. **Interactive R-squared comparison**: A bar chart showing how much variance each model explains, instantly revealing which algorithms capture the popularity signal best.
2. **3D Correlation Surface**: A surface plot mapping the correlation structure between all features and popularity, providing a topographic view of feature relationships.
3. **Interactive Predicted vs. Actual scatter**: For the best-performing model, I plot every prediction against its actual value. Points close to the diagonal line represent accurate predictions; deviations reveal where the model struggles.
4. **Feature Hierarchy (Treemap)**: A treemap visualisation showing how XGBoost allocates its predictive power across features, with larger tiles indicating more important features.

> **Why this section adds value:** While model performance metrics tell us *how accurate* a model is, they do not tell us *what the model learned*. These visualisations bridge that gap, they show which features matter, how predictions compare to reality, and where the correlations are strongest. This transforms raw numbers into actionable musical insights.


In [ ]:
# Model R-squared Comparison (Interactive)
display(Markdown(
    "### 4.1 Model Accuracy (R-squared) Comparison\n"
    "The bar chart below compares R-squared scores across all four models. R-squared measures the proportion of variance "
    "in popularity that each model explains: a higher value means the model captures more of the underlying pattern. "
    "Bars are colour-coded by performance, making it immediately clear which model leads."
))

r2_plot_df = results_df.reset_index().rename(columns={'index': 'Model'})

fig = px.bar(
    r2_plot_df, x='Model', y='R2', color='R2',
    text_auto='.4f',
    title='Model Accuracy Comparison (R-squared Score)',
    template='plotly_dark',
    color_continuous_scale='Viridis'
)
fig.update_layout(yaxis_range=[min(0, r2_plot_df['R2'].min() - 0.1), max(1, r2_plot_df['R2'].max() + 0.1)],
                  width=850, height=500)
fig.update_traces(textposition='outside')
fig.show()

# 3D Correlation Surface
display(Markdown(
    "### 4.2 3D Correlation Surface\n"
    "This surface plot renders the entire correlation matrix as a 3D landscape. Peaks represent strong positive correlations "
    "and valleys represent strong negative correlations. The elevation of each point corresponds to the Pearson coefficient "
    "between the row and column features. Rotate the surface to explore relationships from different angles.\n\n"
    "> **Why this adds value:** A flat heatmap shows correlation values but lacks depth perception. The 3D surface makes it "
    "easier to spot clusters of highly correlated features (plateaus) and dramatic inversions (deep valleys), giving a more "
    "intuitive feel for the multicollinearity structure."
))

corr = df[features + [target]].corr()
fig = go.Figure(data=[go.Surface(
    z=corr.values, x=corr.columns.tolist(), y=corr.columns.tolist(),
    colorscale='RdBu_r', cmin=-1, cmax=1,
    hovertemplate='%{x} vs %{y}<br>Correlation: %{z:.3f}<extra></extra>'
)])
fig.update_layout(
    title='3D Correlation Surface: Audio Features & Popularity',
    template='plotly_dark', width=950, height=700,
    scene=dict(xaxis_title='Features', yaxis_title='Features', zaxis_title='Correlation'),
    margin=dict(l=0, r=0, b=0, t=50)
)
fig.show()

# Interactive Predicted vs Actual (Best Model)
display(Markdown(
    "### 4.3 Predicted vs. Actual Popularity (Best Model)\n"
    "For the best-performing model, I plot every test-set prediction against its actual popularity value. "
    "The white dashed line represents **perfect prediction**, points sitting on this line were predicted exactly right. "
    "Points are coloured by their absolute error (distance from the line), with size proportional to energy. "
    "Hover over any point to see the exact actual vs. predicted values.\n\n"
    "> **Why this adds value:** Aggregate metrics like R-squared hide the distribution of errors. This scatter plot reveals "
    "whether the model systematically over- or under-predicts at certain popularity ranges, and whether errors are random or patterned."
))

best_name = results_df['R2'].idxmax()
y_pred_best = predictions[best_name]

plot_df = X_test.copy()
plot_df['Actual'] = y_test.values
plot_df['Predicted'] = y_pred_best
plot_df['AbsError'] = np.abs(y_test.values - y_pred_best)

fig = px.scatter(
    plot_df, x='Actual', y='Predicted', color='AbsError',
    color_continuous_scale='Reds',
    opacity=0.5,
    title=f'Predicted vs. Actual Popularity: {best_name}',
    labels={'Actual': 'Actual Popularity', 'Predicted': 'Predicted Popularity', 'AbsError': 'Absolute Error'},
    hover_data={'Actual': ':.1f', 'Predicted': ':.1f', 'AbsError': ':.1f'}
)
fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode='lines',
                         line=dict(color='white', dash='dash', width=2), name='Perfect Prediction'))
fig.update_layout(template='plotly_dark', width=850, height=650)
fig.show()

# Feature Hierarchy Treemap
display(Markdown(
    "### 4.4 XGBoost Feature Hierarchy (Treemap)\n"
    "The treemap below visualises how XGBoost distributes its predictive power across features. "
    "Larger tiles represent features that the model relies upon more heavily. Colour intensity reinforces importance ranking.\n\n"
    "> **Why this adds value:** Unlike a simple bar chart, a treemap conveys proportional importance, you can instantly see "
    "that one or two features dominate while others contribute minimally. This is particularly useful for stakeholders who "
    "need a quick, intuitive summary of what drives popularity."
))

if "XGBoost" in feature_importances:
    imp_df = pd.DataFrame({
        'Feature': feature_importances['XGBoost'].index,
        'Importance': feature_importances['XGBoost'].values
    })
    fig = px.treemap(
        imp_df, path=['Feature'], values='Importance',
        color='Importance', color_continuous_scale='Blues',
        title='XGBoost Feature Importance Hierarchy',
        template='plotly_dark'
    )
    fig.update_layout(width=850, height=550)
    fig.show()


### 4.1 Model Accuracy (R-squared) Comparison
The bar chart below compares R-squared scores across all four models. R-squared measures the proportion of variance in popularity that each model explains: a higher value means the model captures more of the underlying pattern. Bars are colour-coded by performance, making it immediately clear which model leads.

### 4.2 3D Correlation Surface
This surface plot renders the entire correlation matrix as a 3D landscape. Peaks represent strong positive correlations and valleys represent strong negative correlations. The elevation of each point corresponds to the Pearson coefficient between the row and column features. Rotate the surface to explore relationships from different angles.

> **Why this adds value:** A flat heatmap shows correlation values but lacks depth perception. The 3D surface makes it easier to spot clusters of highly correlated features (plateaus) and dramatic inversions (deep valleys), giving a more intuitive feel for the multicollinearity structure.

### 4.3 Predicted vs. Actual Popularity (Best Model)
For the best-performing model, I plot every test-set prediction against its actual popularity value. The white dashed line represents **perfect prediction**, points sitting on this line were predicted exactly right. Points are coloured by their absolute error (distance from the line), with size proportional to energy. Hover over any point to see the exact actual vs. predicted values.

> **Why this adds value:** Aggregate metrics like R-squared hide the distribution of errors. This scatter plot reveals whether the model systematically over- or under-predicts at certain popularity ranges, and whether errors are random or patterned.

### 4.4 XGBoost Feature Hierarchy (Treemap)
The treemap below visualises how XGBoost distributes its predictive power across features. Larger tiles represent features that the model relies upon more heavily. Colour intensity reinforces importance ranking.

> **Why this adds value:** Unlike a simple bar chart, a treemap conveys proportional importance, you can instantly see that one or two features dominate while others contribute minimally. This is particularly useful for stakeholders who need a quick, intuitive summary of what drives popularity.

# 5. Diagnosing Model Errors (Residual Analysis)

This section tackles **Objective 2**, closely examining the mistakes each model makes to understand not just *how much* they err, but *where* and *how*. I create four complementary visualisations:

1. **Interactive Residual Scatter Plots**: Residuals (actual minus predicted) plotted against actual popularity. Systematic patterns here indicate model bias, for instance, if residuals trend upward, the model consistently underestimates high-popularity tracks.
2. **Interactive Residual Distributions**: Histograms showing the shape of each model's error distribution. A narrow, symmetric distribution centered at zero is ideal; skew or heavy tails reveal problematic tendencies.
3. **3D Residual Landscape**:  A three-dimensional view mapping actual popularity, predicted popularity, and the residual simultaneously. This reveals whether large errors cluster in specific regions of the prediction space.
4. **Interactive Residual Box Plot Comparison**: Side-by-side box plots comparing the spread, median, and outliers of each model's residuals in a single view.

> **Why this adds value:** Accuracy metrics like RMSE summarise all errors into a single number, hiding important structure. Two models with identical RMSE can have very different error *patterns*, one might underestimate popular songs while the other overestimates everything. Residual analysis exposes these hidden biases, enabling more informed model selection.


In [ ]:
# Interactive Residual Scatter Plots
residuals = {name: y_test.values - pred for name, pred in predictions.items()}

display(Markdown(
    "### 5.1 Residual Scatter Plots\n"
    "Each subplot shows residuals (Actual - Predicted) against actual popularity for one model. "
    "The red dashed line at zero represents perfect prediction. Points above the line indicate the model **underestimated** "
    "popularity; points below mean it **overestimated**. Hover over points for exact values."
))

fig = make_subplots(rows=2, cols=2, subplot_titles=list(residuals.keys()),
                    horizontal_spacing=0.08, vertical_spacing=0.12)

colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']
for i, (name, res) in enumerate(residuals.items()):
    row, col = (i // 2) + 1, (i % 2) + 1
    fig.add_trace(go.Scatter(
        x=y_test.values, y=res, mode='markers',
        marker=dict(color=colors[i], opacity=0.4, size=4),
        name=name,
        hovertemplate='Actual: %{x:.1f}<br>Residual: %{y:.1f}<extra></extra>'
    ), row=row, col=col)
    fig.add_hline(y=0, line_dash="dash", line_color="red", row=row, col=col)

fig.update_layout(template='plotly_dark', width=950, height=750, showlegend=False,
                  title_text='Residual Scatter Plots: All Models')
fig.update_xaxes(title_text='Actual Popularity')
fig.update_yaxes(title_text='Residual (Error)')
fig.show()

# Interactive Residual Distributions
display(Markdown(
    "### 5.2 Residual Distributions\n"
    "These histograms show how residuals are distributed for each model. A narrow bell curve centered at zero is ideal, "
    "it means errors are small, random, and unbiased. Wider distributions indicate inconsistency; "
    "skewed distributions indicate systematic over- or under-prediction."
))

fig = make_subplots(rows=2, cols=2, subplot_titles=list(residuals.keys()),
                    horizontal_spacing=0.08, vertical_spacing=0.12)

for i, (name, res) in enumerate(residuals.items()):
    row, col = (i // 2) + 1, (i % 2) + 1
    fig.add_trace(go.Histogram(
        x=res, nbinsx=40, marker_color=colors[i], opacity=0.75, name=name,
        hovertemplate='Residual: %{x:.1f}<br>Count: %{y}<extra></extra>'
    ), row=row, col=col)
    fig.add_vline(x=0, line_dash="dash", line_color="red", row=row, col=col)

fig.update_layout(template='plotly_dark', width=950, height=750, showlegend=False,
                  title_text='Residual Distributions: All Models')
fig.update_xaxes(title_text='Residual (Error)')
fig.update_yaxes(title_text='Frequency')
fig.show()

# 3D Residual Landscape
display(Markdown(
    "### 5.3 3D Residual Landscape (Best Model)\n"
    "This 3D scatter maps **actual popularity** (x), **predicted popularity** (y), and the **residual** (z) for the best model. "
    "The ideal plane is z=0, all points sitting flat on this plane would mean perfect predictions. "
    "Points rising above or below reveal where large errors occur. Colour intensity reflects absolute error magnitude.\n\n"
    "> **Why this adds value:** The 3D view reveals error clusters that are invisible in 2D. For example, you might see that "
    "large positive residuals cluster at low actual popularity (the model overestimates quiet tracks), creating a visible ridge "
    "in the landscape."
))

best_name = results_df['R2'].idxmax()
best_res = residuals[best_name]

fig = px.scatter_3d(
    x=y_test.values, y=predictions[best_name], z=best_res,
    color=np.abs(best_res),
    color_continuous_scale='Reds',
    opacity=0.5,
    title=f'3D Residual Landscape: {best_name}',
    labels={'x': 'Actual Popularity', 'y': 'Predicted Popularity', 'z': 'Residual', 'color': 'Abs Error'}
)
fig.update_layout(
    template='plotly_dark', width=950, height=700,
    scene=dict(xaxis_title='Actual Popularity', yaxis_title='Predicted Popularity', zaxis_title='Residual (Error)')
)
fig.show()

# Interactive Box Plot Comparison
display(Markdown(
    "### 5.4 Residual Spread Comparison (Box Plot)\n"
    "This box plot directly compares the error distributions of all models side by side. "
    "A smaller box indicates tighter, more consistent predictions. Outliers (dots beyond the whiskers) show where a model "
    "really failed badly. The median line reveals any systematic bias (off-center median = biased model)."
))

residuals_list = []
for name, res in residuals.items():
    temp_df = pd.DataFrame({'Residual': res, 'Model': name})
    residuals_list.append(temp_df)
residuals_box_df = pd.concat(residuals_list)

fig = px.box(
    residuals_box_df, x='Model', y='Residual', color='Model',
    title='Residual Spread Comparison Across Models',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(template='plotly_dark', width=850, height=550, showlegend=False)
fig.show()

display(Markdown(
    "### Conclusion on Error Analysis\n"
    "By analysing residuals, I observe that **Linear Regression** shows systematic patterns, it consistently "
    "underestimates popular songs and overestimates less popular ones, indicating it misses non-linear effects. "
    "**Random Forest** and **XGBoost** have tighter, more balanced error distributions with fewer extreme outliers, "
    "confirming they capture the data's complexity better. "
    "The **Neural Network** performs decently but shows wider tails with occasional large outliers, suggesting it would "
    "benefit from additional tuning or regularisation.\n\n"
    "Overall, **Random Forest and XGBoost** not only have better headline accuracy but also make more consistent, "
    "less biased errors, giving me confidence in their reliability for real-world predictions."
))


### 5.1 Residual Scatter Plots
Each subplot shows residuals (Actual - Predicted) against actual popularity for one model. The red dashed line at zero represents perfect prediction. Points above the line indicate the model **underestimated** popularity; points below mean it **overestimated**. Hover over points for exact values.

### 5.2 Residual Distributions
These histograms show how residuals are distributed for each model. A narrow bell curve centered at zero is ideal, it means errors are small, random, and unbiased. Wider distributions indicate inconsistency; skewed distributions indicate systematic over- or under-prediction.

### 5.3 3D Residual Landscape (Best Model)
This 3D scatter maps **actual popularity** (x), **predicted popularity** (y), and the **residual** (z) for the best model. The ideal plane is z=0, all points sitting flat on this plane would mean perfect predictions. Points rising above or below reveal where large errors occur. Colour intensity reflects absolute error magnitude.

> **Why this adds value:** The 3D view reveals error clusters that are invisible in 2D. For example, you might see that large positive residuals cluster at low actual popularity (the model overestimates quiet tracks), creating a visible ridge in the landscape.

### 5.4 Residual Spread Comparison (Box Plot)
This box plot directly compares the error distributions of all models side by side. A smaller box indicates tighter, more consistent predictions. Outliers (dots beyond the whiskers) show where a model really failed badly. The median line reveals any systematic bias (off-center median = biased model).

### Conclusion on Error Analysis
By analysing residuals, I observe that **Linear Regression** shows systematic patterns, it consistently underestimates popular songs and overestimates less popular ones, indicating it misses non-linear effects. **Random Forest** and **XGBoost** have tighter, more balanced error distributions with fewer extreme outliers, confirming they capture the data's complexity better. The **Neural Network** performs decently but shows wider tails with occasional large outliers, suggesting it would benefit from additional tuning or regularisation.

Overall, **Random Forest and XGBoost** not only have better headline accuracy but also make more consistent, less biased errors, giving me confidence in their reliability for real-world predictions.

# 6. Choosing the Best Model & Understanding Feature Importance

This section combines **Objective 3** (feature importance ranking) and **Objective 8** (accuracy vs. interpretability trade-off). I examine what each model considers the most important features for predicting popularity, then explicitly evaluate the trade-off between model accuracy and transparency.

The visualisations below include:

1. **Interactive Feature Importance Bar Charts**: For each model, a horizontal bar chart ranked by importance. This reveals whether models agree on which features matter most, or whether different algorithms prioritise different sonic qualities.
2. **Radar Chart, Feature Profile Comparison**: A radar/spider chart overlaying all four models' feature importance profiles. Where the profiles align, there is consensus; where they diverge, the models see the data differently.
3. **Interactive Accuracy vs. Error Trade-Off**: A scatter plot positioning each model by its R-squared (accuracy) and RMSE (error magnitude), labelled by name. This visualises the classic bias-variance trade-off and shows which models occupy the best position.

> **Why this adds value:** Feature importance answers the crucial question: *"What makes a song popular?"* But different models answer this question differently based on their assumptions. By comparing importance across model types, I can identify features that are **universally important** (robust findings) versus those that only appear important to specific model architectures (potentially artefacts). The accuracy-vs-interpretability trade-off visualisation then guides the practical decision of which model to deploy.


In [8]:
# 6.1 Feature Importance Bar Charts (All Models)
display(Markdown(
    "### 6.1 Feature Importance Rankings by Model\n"
    "Each subplot shows which audio features the model considers most influential for predicting popularity. "
    "For Linear Regression, these are regression coefficients (can be negative). For Random Forest and XGBoost, "
    "these are split-based importances. For the Neural Network, these are derived from first-layer weight magnitudes.\n\n"
    "Hover over any bar for the exact importance value."
))

fig = make_subplots(rows=2, cols=2, subplot_titles=list(feature_importances.keys()),
                    horizontal_spacing=0.15, vertical_spacing=0.12)

colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']
for i, (name, imps) in enumerate(feature_importances.items()):
    imps_sorted = imps.sort_values(ascending=True)
    row, col = (i // 2) + 1, (i % 2) + 1
    fig.add_trace(go.Bar(
        x=imps_sorted.values, y=imps_sorted.index,
        orientation='h', marker_color=colors[i], name=name,
        hovertemplate='%{y}: %{x:.4f}<extra></extra>'
    ), row=row, col=col)

fig.update_layout(template='plotly_dark', width=1000, height=800, showlegend=False,
                  title_text='Feature Importance Across All Models')
fig.show()

# 6.2 Radar Chart: Feature Profile Comparison
display(Markdown(
    "### 6.2 Radar Chart: Feature Profile Comparison\n"
    "The radar chart overlays normalised feature importance profiles for all models on a single spider diagram. "
    "Each axis represents one feature, and each coloured line represents a model's importance profile. "
    "Where lines overlap, models agree; where they diverge, they see the data differently.\n\n"
    "> **Why this adds value:** The radar chart makes consensus and disagreement immediately visible. If all four models "
    "show high importance for 'loudness' and 'danceability', those features are robust predictors regardless of modelling approach. "
    "Features where only one model shows high importance may be artefacts of that particular algorithm."
))

# Normalise importances to 0-1 for comparison
norm_imps = {}
for name, imps in feature_importances.items():
    abs_imps = imps.abs()
    norm_imps[name] = (abs_imps / abs_imps.max()).values.tolist()

categories = features.copy()

fig = go.Figure()
radar_colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']
for i, (name, vals) in enumerate(norm_imps.items()):
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=categories + [categories[0]],
        fill='toself', fillcolor=f'rgba({int(radar_colors[i][1:3],16)},{int(radar_colors[i][3:5],16)},{int(radar_colors[i][5:7],16)},0.1)',
        line=dict(color=radar_colors[i], width=2),
        name=name,
        hovertemplate='%{theta}: %{r:.3f}<extra></extra>'
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    template='plotly_dark', width=800, height=650,
    title='Feature Importance Profiles: Model Comparison (Normalised)',
    showlegend=True
)
fig.show()

# 6.3 Accuracy vs Error Trade-Off
display(Markdown(
    "### 6.3 Accuracy vs. Error Trade-Off (Objective 8)\n"
    "This scatter plot positions each model by its R-squared score (x-axis, higher is better) and RMSE (y-axis, lower is better). "
    "The ideal model sits in the **bottom-right corner**, high accuracy and low error. Models in the top-left are worst. "
    "Point size is proportional to RMSE for visual emphasis.\n\n"
    "> **Why this adds value:** This directly addresses Objective 8: it visually maps the trade-off between accuracy and "
    "interpretability. Linear Regression occupies a transparent but less accurate position, while tree-based models achieve "
    "higher accuracy at the cost of being harder to explain. Decision-makers can choose their position on this spectrum."
))

fig = px.scatter(
    results_df.reset_index().rename(columns={'index': 'Model'}),
    x='R2', y='RMSE', color='Model', size='RMSE',
    text='Model',
    title='Accuracy vs. Error Trade-Off',
    labels={'R2': 'R-squared (Higher is Better)', 'RMSE': 'RMSE (Lower is Better)'},
    color_discrete_sequence=px.colors.qualitative.Set1,
    size_max=30
)
fig.update_traces(textposition='top center', textfont_size=11)
fig.update_layout(template='plotly_dark', width=850, height=600)
fig.show()

display(Markdown(
    "### Conclusion on Feature Importance\n"
    "From this analysis, **loudness, energy, and danceability** emerge as universally important features across all model types. "
    "These are the sonic ingredients most strongly associated with popularity, louder, more energetic, and more danceable tracks "
    "consistently receive higher popularity scores.\n\n"
    "**Acousticness** shows moderate importance with a negative relationship, more acoustic tracks tend to be less popular on Spotify. "
    "**Instrumentalness** also has a negative impact, as vocal tracks dominate popular playlists.\n\n"
    "On the accuracy-interpretability spectrum:\n"
    "- **Linear Regression** is transparent but underpowereed (low R-squared).\n"
    "- **Random Forest** and **XGBoost** hit the sweet spot, high accuracy with interpretable feature importances.\n"
    "- **Neural Network** captures complexity but is the hardest to explain and shows less stable performance."
))


### 6.1 Feature Importance Rankings by Model
Each subplot shows which audio features the model considers most influential for predicting popularity. For Linear Regression, these are regression coefficients (can be negative). For Random Forest and XGBoost, these are split-based importances. For the Neural Network, these are derived from first-layer weight magnitudes.

Hover over any bar for the exact importance value.

### 6.2 Radar Chart: Feature Profile Comparison
The radar chart overlays normalised feature importance profiles for all models on a single spider diagram. Each axis represents one feature, and each coloured line represents a model's importance profile. Where lines overlap, models agree; where they diverge, they see the data differently.

> **Why this adds value:** The radar chart makes consensus and disagreement immediately visible. If all four models show high importance for 'loudness' and 'danceability', those features are robust predictors regardless of modelling approach. Features where only one model shows high importance may be artefacts of that particular algorithm.

### 6.3 Accuracy vs. Error Trade-Off (Objective 8)
This scatter plot positions each model by its R-squared score (x-axis, higher is better) and RMSE (y-axis, lower is better). The ideal model sits in the **bottom-right corner**: high accuracy and low error. Models in the top-left are worst. Point size is proportional to RMSE for visual emphasis.

> **Why this adds value:** This directly addresses Objective 8: it visually maps the trade-off between accuracy and interpretability. Linear Regression occupies a transparent but less accurate position, while tree-based models achieve higher accuracy at the cost of being harder to explain. Decision-makers can choose their position on this spectrum.

### Conclusion on Feature Importance
From this analysis, **loudness, energy, and danceability** emerge as universally important features across all model types. These are the sonic ingredients most strongly associated with popularity, louder, more energetic, and more danceable tracks consistently receive higher popularity scores.

**Acousticness** shows moderate importance with a negative relationship, more acoustic tracks tend to be less popular on Spotify. **Instrumentalness** also has a negative impact, as vocal tracks dominate popular playlists.

On the accuracy-interpretability spectrum:
- **Linear Regression** is transparent but underpowereed (low R-squared).
- **Random Forest** and **XGBoost** hit the sweet spot, high accuracy with interpretable feature importances.
- **Neural Network** captures complexity but is the hardest to explain and shows less stable performance.

# 7. Stress-Testing with a Synthetic Song

This section addresses **Objective 4**, testing whether the models produce sensible, real-world predictions by creating a hypothetical song with specific audio characteristics and observing what each model predicts.

I go beyond a single prediction by conducting a **sensitivity analysis**: systematically varying one feature (energy) while holding all others constant, then extending this to a **3D sensitivity surface** that varies **both energy and danceability** simultaneously. This reveals:

- Whether models agree or disagree on the synthetic song's expected popularity
- How smoothly each model responds to changes in individual features (smooth curves indicate stable learning; jagged curves suggest overfitting)
- How the interplay between two key features (energy and danceability) jointly influences predicted popularity, visualised as a 3D surface

> **Why this adds value:** Real-world model deployment demands more than accuracy on test data. A model that predicts a song with energy=0.99 has popularity 80, but energy=1.0 has popularity 20 would be dangerously unstable. Sensitivity testing exposes these pathological behaviours before they cause problems in production. The 3D surface additionally reveals whether the model has learned plausible interaction effects between features.


In [ ]:
# 7.1 Synthetic Song: Model Predictions
made_up_song = pd.DataFrame([{
    "danceability": 0.7,
    "energy": 0.8,
    "loudness": -5,
    "speechiness": 0.1,
    "acousticness": 0.2,
    "instrumentalness": 0.0,
    "liveness": 0.15,
    "valence": 0.6,
    "tempo": 120,
    "duration_ms": 210000,
    "key": 5,
    "mode": 1
}])

song_predictions = {}
for name, model in models.items():
    song_predictions[name] = model.predict(made_up_song)[0]

song_pred_df = pd.DataFrame({
    'Model': list(song_predictions.keys()),
    'Predicted Popularity': list(song_predictions.values())
})

display(Markdown(
    "### 7.1 Predicted Popularity for a Synthetic Song\n"
    "I created a made-up song with a profile designed for mainstream appeal: **high energy (0.8)**, **good danceability (0.7)**, "
    "**moderate loudness (-5 dB)**, **low acousticness (0.2)**, and **no instrumentals**. "
    "The bar chart below shows what each model predicts as its popularity score. "
    "Agreement between models strengthens confidence in the prediction; divergence highlights model-specific assumptions."
))

fig = px.bar(
    song_pred_df, x='Model', y='Predicted Popularity', color='Model',
    text_auto='.1f',
    title='Predicted Popularity of Synthetic Song',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(template='plotly_dark', width=800, height=500,
                  yaxis_range=[0, 100], showlegend=False)
fig.update_traces(textposition='outside')
fig.show()

# 7.2 Sensitivity Analysis: Energy
display(Markdown(
    "### 7.2 Energy Sensitivity Analysis\n"
    "I systematically vary only the **energy** of the synthetic song from 0 (very low) to 1 (maximum), "
    "keeping all other features fixed. This reveals how each model responds to changes in a single feature. "
    "Smooth, monotonic curves indicate the model has learned a stable relationship; jagged or erratic curves "
    "suggest overfitting or noise sensitivity."
))

energy_range = np.linspace(0, 1, 30)
preds_by_energy = {name: [] for name in models}

for e in energy_range:
    temp = made_up_song.copy()
    temp["energy"] = e
    for name, model in models.items():
        preds_by_energy[name].append(model.predict(temp)[0])

fig = go.Figure()
line_colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']
for i, (name, preds) in enumerate(preds_by_energy.items()):
    fig.add_trace(go.Scatter(
        x=energy_range, y=preds, mode='lines+markers',
        name=name, line=dict(color=line_colors[i], width=3),
        marker=dict(size=4),
        hovertemplate='Energy: %{x:.2f}<br>Predicted: %{y:.1f}<extra></extra>'
    ))

fig.update_layout(
    title='Predicted Popularity as Energy Varies (All Other Features Fixed)',
    xaxis_title='Energy', yaxis_title='Predicted Popularity',
    template='plotly_dark', width=900, height=550,
    legend=dict(x=0.02, y=0.98)
)
fig.show()

# 7.3 3D Sensitivity Surface: Energy x Danceability
display(Markdown(
    "### 7.3 3D Sensitivity Surface: Energy x Danceability\n"
    "This surface plot extends the sensitivity analysis to two dimensions simultaneously. I vary **energy** (x-axis) "
    "and **danceability** (y-axis) across their full ranges while holding all other features constant, and colour the "
    "surface by the **predicted popularity** (z-axis) from the best-performing model. "
    "Rotate the surface to explore how these two key features jointly influence popularity.\n\n"
    "> **Why this adds value:** Single-feature sensitivity tests reveal marginal effects, but music popularity is driven "
    "by feature *interactions*. A song might need both high energy AND high danceability to be popular, neither alone is "
    "sufficient. This surface reveals those interaction effects as gradients, ridges, and valleys in the prediction landscape."
))

best_model_name = results_df['R2'].idxmax()
best_model = models[best_model_name]

energy_grid = np.linspace(0, 1, 25)
dance_grid = np.linspace(0, 1, 25)
E, D = np.meshgrid(energy_grid, dance_grid)

Z = np.zeros_like(E)
for ei in range(len(energy_grid)):
    for di in range(len(dance_grid)):
        temp = made_up_song.copy()
        temp["energy"] = energy_grid[ei]
        temp["danceability"] = dance_grid[di]
        Z[di, ei] = best_model.predict(temp)[0]

fig = go.Figure(data=[go.Surface(
    z=Z, x=energy_grid, y=dance_grid,
    colorscale='Viridis',
    hovertemplate='Energy: %{x:.2f}<br>Danceability: %{y:.2f}<br>Predicted: %{z:.1f}<extra></extra>',
    colorbar=dict(title='Predicted<br>Popularity')
)])
fig.update_layout(
    title=f'3D Sensitivity Surface: Energy x Danceability ({best_model_name})',
    template='plotly_dark', width=950, height=700,
    scene=dict(xaxis_title='Energy', yaxis_title='Danceability', zaxis_title='Predicted Popularity')
)
fig.show()

# 7.4 Final Comparison Table
display(Markdown("### 7.4 Comprehensive Model Comparison"))

final_comparison = pd.DataFrame({
    "Accuracy (R-squared)": [f"{results_df.loc[n, 'R2']:.4f}" for n in results_df.index],
    "Error (RMSE)": [f"{results_df.loc[n, 'RMSE']:.2f}" for n in results_df.index],
    "Interpretability": ["High", "Medium", "Medium", "Low"],
    "Residual Behaviour": [
        "Systematic bias visible",
        "Well balanced, tight spread",
        "Well balanced, tight spread",
        "Some large outliers"
    ],
    "Feature Insights": [
        "Clear coefficients, easy to explain",
        "Strong non-linear importances",
        "Strong non-linear importances",
        "Approximate, harder to interpret"
    ]
}, index=results_df.index)
final_comparison.index.name = "Model"

display(final_comparison)


display(Markdown(
    "## Overall Conclusion\n\n"
    "### What Makes a Song Popular?\n"
    "The analysis reveals that the strongest predictors of track popularity are **loudness**, **energy**, and **danceability**. "
    "These three attributes were identified as the most important by virtually all models, regardless of algorithm type. "
    "Tracks that are louder, more energetic, and more danceable consistently achieve higher popularity scores on Spotify. "
    "Other features play secondary roles, with **acousticness** and **instrumentalness** showing negative associations, "
    "suggesting that mainstream Spotify audiences prefer produced, vocal-driven tracks over acoustic or instrumental compositions.\n\n"
    "### Which Model Wins?\n"
    "Across all objectives, accuracy, residual analysis, feature importance, cross-validation stability, and synthetic song testing, "
    "the results are clear:\n\n"
    "- **Linear Regression** is the simplest and most transparent, but its inability to capture non-linear patterns limits its accuracy.\n"
    "- **Neural Network** can model complex relationships but proved less stable, harder to interpret, and prone to occasional large errors.\n"
    "- **Random Forest** delivered strong accuracy with balanced residuals, interpretable importances, and consistent cross-validation scores.\n"
    "- **XGBoost** emerged as the **overall winner**, combining the highest accuracy with strong feature importance insights and robust handling of non-linear relationships.\n\n"
    "Between the top two, **XGBoost** is recommended when accuracy is the priority, while **Random Forest** remains an excellent choice "
    "when stability and simplicity are valued. For quick, transparent explanations to non-technical audiences, **Linear Regression** "
    "still has a role despite its lower performance."
))


### 7.1 Predicted Popularity for a Synthetic Song
I created a made-up song with a profile designed for mainstream appeal: **high energy (0.8)**, **good danceability (0.7)**, **moderate loudness (-5 dB)**, **low acousticness (0.2)**, and **no instrumentals**. The bar chart below shows what each model predicts as its popularity score. Agreement between models strengthens confidence in the prediction; divergence highlights model-specific assumptions.

### 7.2 Energy Sensitivity Analysis
I systematically vary only the **energy** of the synthetic song from 0 (very low) to 1 (maximum), keeping all other features fixed. This reveals how each model responds to changes in a single feature. Smooth, monotonic curves indicate the model has learned a stable relationship; jagged or erratic curves suggest overfitting or noise sensitivity.

### 7.3 3D Sensitivity Surface: Energy x Danceability
This surface plot extends the sensitivity analysis to two dimensions simultaneously. I vary **energy** (x-axis) and **danceability** (y-axis) across their full ranges while holding all other features constant, and colour the surface by the **predicted popularity** (z-axis) from the best-performing model. Rotate the surface to explore how these two key features jointly influence popularity.

> **Why this adds value:** Single-feature sensitivity tests reveal marginal effects, but music popularity is driven by feature *interactions*. A song might need both high energy AND high danceability to be popular, neither alone is sufficient. This surface reveals those interaction effects as gradients, ridges, and valleys in the prediction landscape.

### 7.4 Comprehensive Model Comparison

,Accuracy (R-squared),Error (RMSE),Interpretability,Residual Behaviour,Feature Insights
Model,,,,,
Linear Regression,0.0685,23.98,High,Systematic bias visible,"Clear coefficients, easy to explain"
Random Forest,0.2869,20.98,Medium,"Well balanced, tight spread",Strong non-linear importances
XGBoost,0.1773,22.54,Medium,"Well balanced, tight spread",Strong non-linear importances
Neural Network,-0.1813,27.01,Low,Some large outliers,"Approximate, harder to interpret"


## Overall Conclusion

### What Makes a Song Popular?
The analysis reveals that the strongest predictors of track popularity are **loudness**, **energy**, and **danceability**. These three attributes were identified as the most important by virtually all models, regardless of algorithm type. Tracks that are louder, more energetic, and more danceable consistently achieve higher popularity scores on Spotify. Other features play secondary roles, with **acousticness** and **instrumentalness** showing negative associations, suggesting that mainstream Spotify audiences prefer produced, vocal-driven tracks over acoustic or instrumental compositions.

### Which Model Wins?
Across all objectives, accuracy, residual analysis, feature importance, cross-validation stability, and synthetic song testing, the results are clear:

- **Linear Regression** is the simplest and most transparent, but its inability to capture non-linear patterns limits its accuracy.
- **Neural Network** can model complex relationships but proved less stable, harder to interpret, and prone to occasional large errors.
- **Random Forest** delivered strong accuracy with balanced residuals, interpretable importances, and consistent cross-validation scores.
- **XGBoost** emerged as the **overall winner**, combining the highest accuracy with strong feature importance insights and robust handling of non-linear relationships.

Between the top two, **XGBoost** is recommended when accuracy is the priority, while **Random Forest** remains an excellent choice when stability and simplicity are valued. For quick, transparent explanations to non-technical audiences, **Linear Regression** still has a role despite its lower performance.

# 8. Reflection

## The Journey

I was at gym one day listening to music on my headphones when I had a thought. Why is the music I like not as popular as the more mainstream songs of these days? This got me thinking about what actually makes a song popular, and gave me the idea of analysing a Spotify dataset for this assignment.

## What I Learned

 I deepened my understanding of the Python data science ecosystem, from pandas data manipulation to scikit-learn model building to Plotly interactive visualisation.
 I learned how different model architectures (linear, ensemble, gradient boosting, neural networks) bring different strengths and weaknesses. Understanding residual analysis and cross-validation gave me tools to evaluate models beyond simple accuracy scores.
 Transitioning from static matplotlib charts to interactive Plotly visualisations taught me how interactivity (hover, zoom, rotate) transforms data exploration from a passive viewing experience into an active investigation.

## Looking Forward

Now when I hear a new song, I will pay attention to its energy, loudness, and danceability, and mentally predict whether it will chart. These analysis assignments are probably my favourite because they feel the most applicable to real-world scenarios. The skills I have developed here, data cleaning, model selection, visual storytelling, and critical evaluation, are foundational to any data-driven career. Overall, I enjoyed this assignment, learned a great deal, and look forward to applying these techniques to new domains.
